In [ ]:
import pandas as pd
import numpy as np

# 1. Subject registry metadata (TSV)
subjects_tsv = """subject_id\thand_dominance\tage_group
S101\tRight\tAdult
S102\tRight\tAdult
S103\tLeft\tYouth
S104\tRight\tSenior
S105\tLeft\tAdult
"""
with open("subjects.tsv", "w") as f:
    f.write(subjects_tsv)

# 2. Raw behavioral trial events (CSV with dirty channel text and missing values)
trials_csv = """trial_id,subject_id,cue,lead_channel,rt_ms,artifact_flag
1001,S101,left_hand, C3.. ,480.5,False
1002,S102,right_hand,Cz... ,512.0,False
1003,S101,right_hand, C4 ,nan,False
1004,S103,feet,Cz.. ,620.3,False
1005,S999,left_hand,C3. ,430.0,False
1006,S104,right_hand, Fz.. ,590.2,True
1007,S102,left_hand,Cz ,nan,True
1008,S105,feet, C4.. ,540.8,False
1009,S101,left_hand,C3 ,460.1,False
1010,S103,right_hand,Cz. ,505.4,False
"""
with open("raw_trials.csv", "w") as f:
    f.write(trials_csv)

In [ ]:
df_subjects = pd.read_csv('subjects.tsv', sep=r'\t')
df_trials = pd.read_csv("raw_trials.csv", sep=r',')
df_trials['lead_channel'] = df_trials["lead_channel"].str.strip(' .')
trial_count = df_trials['cue'].value_counts()
df_merged = pd.merge(df_trials, df_subjects, on='subject_id', how='left')
df_merged['unregistered_subject'] = ~df_merged['subject_id'].isin(df_subjects['subject_id'])
df_merged = df_merged[(~df_merged["artifact_flag"]) & (~df_merged["unregistered_subject"])]


C:\Users\bilal\AppData\Local\Temp\ipykernel_38516\2442363962.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df_subjects = pd.read_csv('subjects.tsv', sep=r'\t')


,trial_id,subject_id,cue,lead_channel,rt_ms,artifact_flag,hand_dominance,age_group,unregistered_subject,target
0,1001,S101,left_hand,C3,480.5,False,Right,Adult,False,0
1,1002,S102,right_hand,Cz,512.0,False,Right,Adult,False,1
2,1003,S101,right_hand,C4,508.7,False,Right,Adult,False,1
3,1004,S103,feet,Cz,620.3,False,Left,Youth,False,2
7,1008,S105,feet,C4,540.8,False,Left,Adult,False,2
8,1009,S101,left_hand,C3,460.1,False,Right,Adult,False,0
9,1010,S103,right_hand,Cz,505.4,False,Left,Youth,False,1


In [83]:
df_merged = df_merged.fillna({'rt_ms': df_merged['rt_ms'].median()})

correspondance = {
    "left_hand": 0,
    "right_hand": 1,
    "feet": 2
}
df_merged['target'] = df_merged['cue'].map(correspondance)
df_merged

,trial_id,subject_id,cue,lead_channel,rt_ms,artifact_flag,hand_dominance,age_group,unregistered_subject,target
0,1001,S101,left_hand,C3,480.5,False,Right,Adult,False,0
1,1002,S102,right_hand,Cz,512.0,False,Right,Adult,False,1
2,1003,S101,right_hand,C4,508.7,False,Right,Adult,False,1
3,1004,S103,feet,Cz,620.3,False,Left,Youth,False,2
7,1008,S105,feet,C4,540.8,False,Left,Adult,False,2
8,1009,S101,left_hand,C3,460.1,False,Right,Adult,False,0
9,1010,S103,right_hand,Cz,505.4,False,Left,Youth,False,1


In [ ]:
df_summary = df_merged.groupby(['age_group', 'cue']).agg(
    mean_rt = ('rt_ms', 'mean'),
    std_rt = ('rt_ms', 'std'),
    n_trials = ('trial_id', 'nunique'),
    ).sort_values(by=('mean_rt'))

y_values = df_merged['target'].to_numpy(dtype="int64")

print(y_values.shape)
print(y_values)

(7,)
[0 1 1 2 2 0 1]


mean_rt     std_rt  n_trials
age_group cue                                     
Adult     left_hand    470.30  14.424978         2
Youth     right_hand   505.40        NaN         1
Adult     right_hand   510.35   2.333452         2
          feet         540.80        NaN         1
Youth     feet         620.30        NaN         1